# Silver to Gold: Building BI-Ready Fact Table

This notebook reads the cleaned silver-layer `slv_order_items` table and enriches it with derived columns to create a BI-ready gold-layer fact table. New columns include monetary calculations (gross amount, discount amount, sale amount), a date key, a coupon flag, and INR currency conversion via FX rate lookup. The final table is written to `ecommerce.gold.gld_fact_order_items`.

**Pipeline context:** Bronze (raw) → Silver (cleaned & typed) → **Gold (BI-ready, enriched)**

## Source and Target

| Source | Target | Rows |
| --- | --- | --- |
| `ecommerce.silver.slv_order_items` | `ecommerce.gold.gld_fact_order_items` | 183,378 |

## Column Rename Mapping

Several silver columns are renamed in the gold table for BI readability:

| Silver Column | Gold Column | Reason |
| --- | --- | --- |
| `dt` | `transaction_date` | Clearer business name |
| `order_ts` | `transaction_ts` | Clearer business name |
| `order_id` | `transaction_id` | Clearer business name |
| `item_seq` | `seq_no` | Shorter, conventional name |
| `discount_pct` | `discount_percent` | Full word for readability |
| `sale_amount` | `net_amount` | Standard finance terminology |
| `sale_amount_inr` | `net_amount_inr` | Matches `net_amount` rename |

All other silver columns retain their original names in the gold table.

In [0]:
# --- Section 1: Setup & Data Loading ---
# Import PySpark SQL functions and type classes needed for column transformations
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType
import pyspark.sql.functions as F

In [0]:
# Define the Unity Catalog name used for reading (silver) and writing (gold) tables
catalog_name = 'ecommerce'

In [0]:
# Read the cleaned silver-layer order items table
# Source: ecommerce.silver.slv_order_items
df_silver = spark.read.table(f"{catalog_name}.silver.slv_order_items")

# Preview the data
df_silver.display()

## Derived Columns Plan

We will add the following columns to the silver table to make it BI-ready:

| # | Column | Description |
|---|--------|-------------|
| 1 | `gross_amount` | quantity × unit_price — total before discount |
| 2 | `discount_amount` | gross_amount × discount_pct / 100 — dollar value of discount |
| 3 | `sale_amount` | gross_amount − discount_amount + tax_amount — final amount owed |
| 4 | `date_id` | date in YYYYMMDD format (derived from `dt`) for joining to date dimension |
| 5 | `coupon_flag` | 1 if a coupon was applied, 0 otherwise |
| 6 | `inr_rate` | FX conversion rate to INR, looked up by currency code |
| 7 | `sale_amount_inr` | sale_amount × inr_rate — sale amount converted to INR |

In [0]:
# --- Section 2: Add Derived Columns ---
# 2.1  gross_amount = quantity × unit_price (total before any discount)
df_silver = df_silver.withColumn(
    "gross_amount",
    (F.col("quantity") * F.col("unit_price"))
)

In [0]:
# 2.2  discount_amount = gross_amount × discount_pct / 100 (dollar value of discount)
df_silver = df_silver.withColumn(
    "discount_amount",
    (F.col("gross_amount") * F.col("discount_pct") / 100)
)

In [0]:
# 2.3  sale_amount = gross_amount - discount_amount + tax_amount (final amount owed)
df_silver = df_silver.withColumn(
    "sale_amount",
    (F.col("gross_amount") - F.col("discount_amount") + F.col("tax_amount"))
)

In [0]:
# 2.4  date_id = dt with hyphens removed (YYYYMMDD format for date dimension joins)
df_silver = df_silver.withColumn(
    "date_id",
    F.regexp_replace(F.col("dt"), "-", "")
)

In [0]:
# 2.5  coupon_flag = 1 if coupon_code is not 'NA', else 0
df_silver = df_silver.withColumn(
    "coupon_flag",
    F.when(F.col("coupon_code") != 'NA', F.lit(1)).otherwise(F.lit(0))
)

In [0]:
# 2.6  inr_rate — FX conversion rate to Indian Rupees (INR)
# Build a lookup DataFrame from a static FX rate dictionary

# FX rates: 1 unit of currency = N INR
fx_rates = {
    "INR": 1.00,
    "AED": 24.18,
    "AUD": 57.55,
    "CAD": 62.93,
    "GBP": 117.98,
    "SGD": 68.18,
    "USD": 88.29,
}

# Convert the dictionary to a list of tuples (currency, inr_rate)
rates = [(k, float(v)) for k, v in fx_rates.items()]

# Create a Spark DataFrame from the rates for joining with the main table
df_rates = spark.createDataFrame(rates, ['currency', 'inr_rate'])
df_rates.show(10)

In [0]:
# Left join the FX rates DataFrame onto the main table
# Join condition: trimmed unit_price_currency = trimmed currency (to handle any whitespace)
# Drop the redundant 'currency' column after the join
df_silver = df_silver.join(df_rates, on=F.trim(df_silver.unit_price_currency) == F.trim(df_rates.currency), how='left').drop('currency')

# Preview the joined data with inr_rate
df_silver.display()

In [0]:
# 2.7  sale_amount_inr = sale_amount × inr_rate (convert final amount to INR)
df_silver = df_silver.withColumn(
    "sale_amount_inr",
    (F.col("sale_amount") * F.col("inr_rate"))
)

In [0]:
# Preview 10 rows to verify all derived columns are populated correctly
df_silver.limit(10).display()

In [0]:
# --- Section 3: Build Final Gold Table ---
# Select and rename columns for the gold-layer fact table
# Renames: dt→transaction_date, order_ts→transaction_ts, order_id→transaction_id,
#          item_seq→seq_no, discount_pct→discount_percent,
#          sale_amount→net_amount, sale_amount_inr→net_amount_inr

orders_gold_df = df_silver.select(
    F.col("date_id"),
    F.col("dt").alias("transaction_date"),
    F.col("order_ts").alias("transaction_ts"),
    F.col("order_id").alias("transaction_id"),
    F.col("customer_id"),
    F.col("item_seq").alias("seq_no"),
    F.col("product_id"),
    F.col("channel"),
    F.col("coupon_code"),
    F.col("coupon_flag"),
    F.col("unit_price_currency"),
    F.col("quantity"),
    F.col("unit_price"),
    F.col("gross_amount"),
    F.col("discount_pct").alias("discount_percent"),
    F.col("discount_amount"),
    F.col("tax_amount"),
    F.col("sale_amount").alias("net_amount"),
    F.col("sale_amount_inr").alias("net_amount_inr")
)

In [0]:
# Preview 50 rows of the final gold-layer table
orders_gold_df.show(50)

In [0]:
# --- Section 4: Write to Gold Layer ---
# Write the final DataFrame to the gold-layer Delta table
# Target: ecommerce.gold.gld_fact_order_items
# Using overwrite mode with mergeSchema to allow schema evolution
orders_gold_df.write.format("delta") \
                    .mode("overwrite") \
                    .option("mergeSchema", True) \
                    .saveAsTable(f"{catalog_name}.gold.gld_fact_order_items")

In [0]:
# Validation: check row count to ensure no data loss during transformation
# Expected: 183,378 rows (matching the silver table)
spark.sql(f"SELECT count(*) AS Total_Row_Count FROM {catalog_name}.gold.gld_fact_order_items").show()

## Summary

The silver-layer order items table has been enriched and written to the Gold layer:

| Property | Value |
| --- | --- |
| Gold Table | `ecommerce.gold.gld_fact_order_items` |
| Source | `ecommerce.silver.slv_order_items` |
| Total Rows | 183,378 (validated — matches silver) |
| Write Mode | `overwrite` with `mergeSchema=True` |

### Derived Columns Added

| Column | Formula | Description |
| --- | --- | --- |
| `gross_amount` | `quantity × unit_price` | Total before discount |
| `discount_amount` | `gross_amount × discount_pct / 100` | Dollar value of discount |
| `sale_amount` | `gross_amount − discount_amount + tax_amount` | Final amount owed |
| `date_id` | `dt` with hyphens removed (YYYYMMDD) | Integer join key for date dimension |
| `coupon_flag` | `1 if coupon_code ≠ 'NA', else 0` | Binary indicator for coupon usage |
| `inr_rate` | FX lookup by `unit_price_currency` | Conversion rate to INR |
| `sale_amount_inr` | `sale_amount × inr_rate` | Sale amount converted to INR |

### FX Rates Used

| Currency | INR Rate |
| --- | --- |
| INR | 1.00 |
| AED | 24.18 |
| AUD | 57.55 |
| CAD | 62.93 |
| GBP | 117.98 |
| SGD | 68.18 |
| USD | 88.29 |

The gold fact table is now ready for BI dashboard consumption and joins with the gold dimension tables (`gld_dim_date` on `date_id`, `gld_dim_products` on `product_id`, `gld_dim_customers` on `customer_id`).